In [ ]:
"""
FSPro input build test
======================
Demonstrates fb_tools weather pipeline → build_fspro_inputs() → run_fspro()
using the 416 sample LCP / ignition shapefile that ships with TestFSPro CLI build.
"""

import numpy as np
from pathlib import Path

from fb_tools import build_fspro_inputs, run_fspro
from fb_tools.weather import (
    load_pyrome_wind_cells,
    load_gridmet_csv,
    load_gridmet_pyrome_cache,
)
from fb_tools.weather.gridmet import (
    build_erc_stats,
    build_erc_classes,
    build_current_erc_values,
)

projdir = Path.cwd().parents[1]
print(f"Project directory: {projdir}")

# --- Set up the model parameters and directories
PYROME_ID  = "47"   # swap to the pyrome that covers your real LCP

# Executable + sample data (416 Fire, La Plata County CO)
FSPRO_EXE  = projdir / "code" / "FB" / "bin" / "TestFSPro.exe"
SAMPLE_DIR = projdir / "code" / "FB" / "TestFSPro" / "SampleData"
LCP        = SAMPLE_DIR / "416lcp.lcp"
IGN_SHP    = SAMPLE_DIR / "416ign.shp"

# Weather inputs
CSV_PATH  = projdir / "data" / "tabular" / "raw" / "weather" / "gridmet_clima_co_pyromes.csv"
WIND_DIR  = projdir / "data" / "weather" / "pyrome_wind"
ERC_DIR   = projdir / "data" / "weather" / "pyrome_erc"

OUT_DIR   = projdir / "data" / "fspro_test" / "build_test"

print()
for p in [FSPRO_EXE, LCP, IGN_SHP, CSV_PATH]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

## 1. Load pyrome weather cache

Wind cells from the HRRR run and historic ERC from the GridMET cache are both
already on disk as JSON — no re-download needed.

In [ ]:
# ── Wind climatology ──────────────────────────────────────────────────────────
wind_meta  = load_pyrome_wind_cells(PYROME_ID, WIND_DIR, return_meta=True)
wind_cells = np.array(wind_meta["WindCellValues"])
calm_value = wind_meta["CalmValue"]

print(f"Pyrome {PYROME_ID} wind cells: {wind_cells.shape}  calm={calm_value:.2f}%")
print(wind_cells)

# ── Historic ERC ──────────────────────────────────────────────────────────────
erc_meta     = load_gridmet_pyrome_cache(PYROME_ID, ERC_DIR, return_meta=True)
erc_historic = np.array(erc_meta["HistoricERCValues"])

print(f"\nHistoric ERC: {erc_historic.shape}  "
      f"({erc_meta['NumERCYears']} years × {erc_meta['NumWxPerYear']} season-days)")

## 2. ERC statistics and current-season ERC

`build_erc_stats` computes per-DOY mean/std across years (→ `AvgERCValues` / `StdDevERCValues`).
`build_current_erc_values` uses the column-wise median as a climatological proxy for
`CurrentERCValues` — appropriate for scenario runs where we are not targeting a specific date.

In [ ]:
historic_dict = {PYROME_ID: erc_historic}

stats   = build_erc_stats(historic_dict)
erc_avg = stats[PYROME_ID]["avg"]
erc_std = stats[PYROME_ID]["std"]

# start_doy is fire-season-relative: 1 = Apr 1, 60 = Jun 1, 122 = Aug 1
start_doy   = 60   # climatological mid-season start
current_erc = build_current_erc_values(historic_dict, start_doy=start_doy)[PYROME_ID]

print(f"ERC avg/std shape : {erc_avg.shape}")
print(f"Current ERC       : {len(current_erc)} days from DOY {start_doy}")
print(f"  min={current_erc.min():.0f}  "
      f"median={np.median(current_erc):.0f}  "
      f"max={current_erc.max():.0f}")

## 3. ERC classes

Quintile ERC bins with NFDRS-derived fuel moisture values per class.
Requires the GridMET CSV (produced by `00a-GridMET_Climatology-GEE.ipynb`).

In [ ]:
clim        = load_gridmet_csv(CSV_PATH)
classes     = build_erc_classes(clim)
erc_classes = classes[PYROME_ID]

print(f"ERC classes shape: {erc_classes.shape}  (5 classes × 10 cols)")
print()
print(f"  {'lower':>6} {'upper':>6} {'fm1':>5} {'fm10':>5} {'fm100':>6} "
      f"{'fm_herb':>7} {'fm_woody':>8} {'spot_d':>7} {'spot_p':>7} {'spot2':>6}")
for row in erc_classes:
    print(f"  {row[0]:>6.0f} {row[1]:>6.0f} {row[2]:>5.1f} {row[3]:>5.1f} "
          f"{row[4]:>6.1f} {row[5]:>7.1f} {row[6]:>8.1f} "
          f"{row[7]:>7.0f} {row[8]:>7.2f} {row[9]:>6.0f}")

## 4. Build the FSPro input file

All weather blocks are now in hand. `build_fspro_inputs()` writes the complete
`FSPRO-Inputs-File-Version-4` text file.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

input_path = build_fspro_inputs(
    output_path   = OUT_DIR / f"pyrome_{PYROME_ID}_test.input",
    wind_cells    = wind_cells,
    calm_value    = calm_value,
    erc_historic  = erc_historic,
    erc_avg       = erc_avg,
    erc_std       = erc_std,
    erc_classes   = erc_classes,
    current_erc   = current_erc,
    ignition_file = IGN_SHP,
    NumFires      = 100,   # small for test speed; production: 1000+
    Duration      = 7,
)

print(f"\nWritten: {input_path}")
print(f"  Size : {input_path.stat().st_size / 1024:.1f} KB")

## 5. Spot-check the generated input file

Print the header + wind rose + ERC class block.
Skip the bulk `HistoricERCValues` body (15 rows × 214 values).

In [ ]:
text  = input_path.read_text()
lines = text.splitlines()

# Print up to and including the HistoricERCValues header, then skip body
skip_body   = False
resume_at   = "AvgERCValues"
for line in lines:
    if skip_body:
        # Resume printing once we hit the next named section
        if line.startswith(resume_at):
            skip_body = False
        else:
            continue
    print(line)
    if line.startswith("HistoricERCValues"):
        print(f"  ... ({erc_meta['NumERCYears']} rows × {erc_meta['NumWxPerYear']} values — truncated)")
        skip_body = True

---
## 6. Run FSPro  *(Windows VM only)*

The cells below call `TestFSPro.exe` and will raise `RuntimeError` on macOS.
Run them in the Windows VM (Parallels) where the path resolves via the shared Box drive.

In [ ]:
result = run_fspro(
    fspro_exe        = FSPRO_EXE,
    lcp_fp           = LCP,
    input_file       = input_path,
    output_directory = OUT_DIR,
    output_basename  = f"fspro_p{PYROME_ID}",
    num_fires_warn   = 100,
)

print(f"Return code: {result.returncode}")

In [ ]:
outputs = sorted(OUT_DIR.glob(f"fspro_p{PYROME_ID}*"))
print(f"{len(outputs)} output file(s):")
for f in outputs:
    print(f"  {f.name:55s}  {f.stat().st_size / 1024:8.1f} KB")

In [ ]:
import rioxarray as rxr
import matplotlib.pyplot as plt

bp_asc = OUT_DIR / f"fspro_p{PYROME_ID}_BurnProb.asc"

if bp_asc.exists():
    # isel(band=0) — do NOT use squeeze("band"); multi-band TIFs will raise ValueError
    da = rxr.open_rasterio(bp_asc, masked=True).isel(band=0)

    fig, ax = plt.subplots(figsize=(7, 6))
    da.plot(ax=ax, cmap="YlOrRd", cbar_kwargs={"label": "Burn Probability"})
    ax.set_title(f"FSPro Burn Probability\nPyrome {PYROME_ID} weather / 416 sample LCP")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
else:
    print("BurnProb.asc not found — FSPro may not have run yet, or check the log:")
    log = OUT_DIR / "TestFSPro_run.log"
    if log.exists():
        print(log.read_text()[-1500:])